***Laue Pattern peak search on a large set of binary Images using multiprocessing***

on jupyter-slurm.esrf.fr

or

locally on desktop computer (with some cpus) and with access to the LaueTools environment or folder

or lbm32gpu1  (@ESRF)

**This Notebook is a part of the LaueTools Package** 
Author: J.-S. Micha

Last Revision:   January 2026

tested with python 3.12  on jupyterlab and jupyter hub, jupyter-notebook

**Objectives**

- Load and display Laue pattern images
- Perform a Peak Search, display found peaks
- Perform a Peak Search on several images with several cpus

In [ ]:
!which python

# SET runtime jupyterlab or not

In [ ]:
JUPYTER_LAB = True # jupyter lab (at ESRF)
JUPYTER_HUB = False # jupyter hub or notebook
JUPYTER_VSCODE = False #  jupyter on vscode

# imports

**check number of available cores**

In [ ]:
import multiprocessing
import itertools
nb_cpus = multiprocessing.cpu_count()
print(f"There are {nb_cpus} cores available - setting the variable nb_cpus to use them all ")

In [ ]:
if JUPYTER_HUB : # jupyterhub
    %matplotlib notebook
elif JUPYTER_VSCODE: # vscode
    %matplotlib widget
elif JUPYTER_LAB: #  jupyterlab
    %matplotlib widget


import time,copy,os
from pathlib import Path
from tqdm import tqdm, tqdm_notebook

import multiprocessing
import itertools
from multiprocessing import active_children, cpu_count
nb_cpus = multiprocessing.cpu_count()
print(f"There are {nb_cpus} cores available - setting the variable nb_cpus to use them all ")

from ipywidgets import interact, interactive, fixed, interact_manual

# Third party modules
import matplotlib as mpl     # graphs and plots
import matplotlib.pyplot as plt
import numpy as np    # numerical arrays
import fabio
from matplotlib.patches import Ellipse
# matplotlib preferences
plt.style.use('classic')
mpl.rcParams['image.cmap'] = 'inferno'
mpl.rcParams['figure.facecolor'] = 'none'
mpl.rcParams['font.size'] = 8

In [ ]:
#OR use your own lauetools.
# Setting absolute path to LaueTools Modules if Lauetools has not been installed with pip
if 0: 
    import sys
    # for slurm machines
    # sys.path.insert(0,'/home/esrf/micha/lauetools_devNotebooks/lauetools')
    # # for lbm32gpu1 machine (test temp)
    #sys.path.insert(0,'/data/bm32/inhouse/STAFF/JSM/lauetools_devNotebooks/lauetools')

    sys.path.insert(0,'/data/bm32/inhouse/lauetoolsenv2/lib/python3.12/site-packages')
    
    import LaueTools as LT
    print('code from', LT.__file__)

In [ ]:
# LaueTools modules
from LaueTools import IOLaueTools as IOLT   # read and write ASCII file  (IO) 
from LaueTools import readmccd as RMCCD       # read CCD and detector binary file, PeakSearch methods
from LaueTools import lauecore as LAUE       
from LaueTools import CrystalParameters as CP       
from LaueTools import generaltools as GT
from LaueTools import LaueGeometry as LaueGeo
from LaueTools import dict_LaueTools as DictLT
from LaueTools import imageprocessing as ImProc


import LaueTools as LT
print('Using lauetools code in ', LT.__file__)

# set user data filename & folders, detector type

In [ ]:
expId = '322812'
expId = 'blc15488'
expId = 'utr20'
expId = 'ma6758'
expId='blc17163'
expId = 'a321220'  # guinebretiere MgO

In [ ]:
'/data/visitor/ma6758/bm32/20260312/RAW_DATA/Al/Al_map2D/scan0006/'

In [ ]:
'/data/visitor/a321220/bm32/20260915/RAW_DATA/MH73_HT/MH73_HT_map_2D_p7_MgO_Fe_800/scan0001/eiger4m_0006.h5

In [ ]:
if expId == 'a321220':
    imageindex = 0 # Al
 #    {'scantype': 'map',
 # 'start_time': Timestamp('2026-09-16 22:51:06'),
 # 'end_time': Timestamp('2026-09-17 05:41:28'),
 # 'sample_dataset_scanindex': 'MH73_HT_montee_500_5',
 # 'fullcommand': 'amesh yech -6.43556 -6.28556 150 xech 0.94696 1.09696 150 0.5',
 # 'scanindex': '5',
 # 'motors': 'xech yech',
 # 'localhdf5file': '/data/visitor/a321220/bm32/20260915/RAW_DATA/MH73_HT/MH73_HT_montee_500/MH73_HT_montee_500.h5',
 # 'imagefolder': '/data/visitor/a321220/bm32/20260915/RAW_DATA/MH73_HT/MH73_HT_montee_500/scan0005',
 # 'endreason': 'SUCCESS',
 # 'samplename': b'MH73_HT',
 # 'folder': '/data/visitor/a321220/bm32/20260915/RAW_DATA/MH73_HT/MH73_HT_montee_500/scan0005',
 # 'nodeinhdf5file': 'MH73_HT/MH73_HT_montee_500/5.1',
 # 'prefix': 'eiger4m_',
 # 'suffix': 'h5',
 # 'listindices': array([    0,     1,     2, ..., 22798, 22799, 22800], shape=(22801,)),
 # 'nbimagesperline': 151,
 # 'mapdimensions': (151, 151),
 # 'peaklistfile': None,
 # 'fastaxis': 'yech',
 # 'slowaxis': 'xech',
 # 'collector': 'pixelval',
 # 'CCDLabel': 'EIGER_4MCdTe'}
    maindatafolder = '/data/visitor/a321220/bm32/20260915/RAW_DATA'
    #subfolderscandata = 'MH73_HT/MH73_HT_montee_500/scan0005'

    subfolderscandata = 'MH73_HT/MH73_HT_montee_500/scan0005'
    subfolderscandata = 'MH73_HT/MH73_HT_map_2D_p7_MgO_Fe_800/scan0001'
    
    imagefolder = Path(maindatafolder+'/'+subfolderscandata)
    
    # calibration parameters file
    #'/data/bm32/inhouse/data/laue/32-02-812/calibGe111_sCMOS_3202812_vendredi_ech44.det'
    detfile = Path('/data/visitor/a321220/bm32/20260915/RAW_DATA/MH73/MH73_Ge/scan0001/calibGe001_MH73_RT_mardi.det')
    
    mainprocessed_data = Path(str(Path(maindatafolder).parent)+'//'+'PROCESSED_DATA')
    # Create the folder if it doesn't exist
    mainprocessed_data.mkdir(parents=True, exist_ok=True)
    datfilefolder = Path(str(mainprocessed_data)+'//'+subfolderscandata+'//'+'corfiles')
    corfilefolder = datfilefolder
     # Make sure directory exists
    os.makedirs(datfilefolder, exist_ok=True)

    prefixfilename= 'eiger4m_'
    suffix='.h5'
    CCDLabel = 'EIGER_4MCdTe'
    sizeofzeropadding = 4


if expId == 'blc17163':
    imageindex = 0 # Al
   
    maindatafolder = '/data/visitor/blc17163/bm32/20260609/RAW_DATA/'
    subfolderscandata = 'BaTiO3/BaTiO3_Map/scan0001/'
    
    imagefolder = Path(maindatafolder+'/'+subfolderscandata)
    
    # calibration parameters file
    #'/data/bm32/inhouse/data/laue/32-02-812/calibGe111_sCMOS_3202812_vendredi_ech44.det'
    detfile = Path('/data/visitor/ma6758/bm32/20260312/RAW_DATA/Al/Al_Gealignwire/scan0003/CalibGe001_eiger4m_zcam100mm.det')
    
    mainprocessed_data = Path(str(Path(maindatafolder).parent)+'//'+'PROCESSED_DATA')
    # Create the folder if it doesn't exist
    mainprocessed_data.mkdir(parents=True, exist_ok=True)
    datfilefolder = Path(str(mainprocessed_data)+'//'+subfolderscandata+'//'+'corfiles')
    corfilefolder = datfilefolder
     # Make sure directory exists
    os.makedirs(datfilefolder, exist_ok=True)

    prefixfilename= 'eiger4m_'
    suffix='.h5'
    CCDLabel = 'EIGER_4MCdTe'
    sizeofzeropadding = 4


if expId == 'ma6758':
    imageindex = 0 # Al
   
    maindatafolder = '/data/visitor/ma6758/bm32/20260312/RAW_DATA'
    subfolderscandata = '/Al/Al_map2D/scan0006'
    subfolderscandata = 'Al/Al_Al650um_daxm_h1.7_expo_5_nbsteps_650_hmicro1p7mm/scan0002'
    #subfolderscandata = 'ZrO2/ZrO2_ZrO2_1330C/scan0002'
    imagefolder = Path(maindatafolder+'/'+subfolderscandata)
    
    # calibration parameters file
    #'/data/bm32/inhouse/data/laue/32-02-812/calibGe111_sCMOS_3202812_vendredi_ech44.det'
    detfile = Path('/data/visitor/ma6758/bm32/20260312/RAW_DATA/Al/Al_Gealignwire/scan0003/CalibGe001_eiger4m_zcam100mm.det')
    
    mainprocessed_data = Path(str(Path(maindatafolder).parent)+'//'+'PROCESSED_DATA')
    # Create the folder if it doesn't exist
    mainprocessed_data.mkdir(parents=True, exist_ok=True)
    datfilefolder = Path(str(mainprocessed_data)+'//'+subfolderscandata+'//'+'corfiles')
    corfilefolder = datfilefolder
     # Make sure directory exists
    os.makedirs(datfilefolder, exist_ok=True)

    prefixfilename= 'eiger4m_'
    suffix='.h5'
    CCDLabel = 'EIGER_4MCdTe'
    sizeofzeropadding = 4

# utr 20 hercules al
if expId == 'utr20':
    imageindex = 0 # Al
   
    maindatafolder = '/data/visitor/utr20/bm32/20260310'
    subfolderscandata = '/RAW_DATA/Alafternoon/Alafternoon_map2D/scan0001'
    #subfolderscandata = 'ZrO2/ZrO2_ZrO2_1330C/scan0002'
    imagefolder = Path(maindatafolder+'/'+subfolderscandata)
    
    # calibration parameters file
    #'/data/bm32/inhouse/data/laue/32-02-812/calibGe111_sCMOS_3202812_vendredi_ech44.det'
    detfile = Path('/data/visitor/utr20/bm32/20260310/RAW_DATA/Alafternoon/Alafternoon_Ge/scan0001/calib_Ge_001_EIGER4M.det')
    
    mainprocessed_data = Path(maindatafolder+'//'+'PROCESSED_DATA')
    # Create the folder if it doesn't exist
    mainprocessed_data.mkdir(parents=True, exist_ok=True)
    datfilefolder = Path(str(mainprocessed_data)+'//'+subfolderscandata+'//'+'corfiles')
    corfilefolder = datfilefolder
     # Make sure directory exists
    os.makedirs(datfilefolder, exist_ok=True)

    prefixfilename= 'eiger4m_'
    suffix='.h5'
    CCDLabel = 'EIGER_4MCdTe'
    sizeofzeropadding = 4
    
    
if expId == '322812':
    imageindex = 0 # Al
    imageindex = 6 # Al2Cu
    
    maindatafolder = '/data/bm32/inhouse/data/laue/32-02-812'
    subfolderscandata = ''
    #subfolderscandata = 'ZrO2/ZrO2_ZrO2_1330C/scan0002'
    imagefolder = Path(maindatafolder+'/'+subfolderscandata)
    
    # calibration parameters file
    #'/data/bm32/inhouse/data/laue/32-02-812/calibGe111_sCMOS_3202812_vendredi_ech44.det'
    detfile = Path(maindatafolder+'/'+'calibGe111_sCMOS_3202812_vendredi_ech44.det')
    
    #mainprocessed_data = Path(maindatafolder+'//'+'PROCESSED_DATA')
    # Define the folder path starting from the home directory
    mainprocessed_data = Path("~/LaueTutorialsResults").expanduser()
    
    # Create the folder if it doesn't exist
    mainprocessed_data.mkdir(parents=True, exist_ok=True)
    
    datfilefolder = Path(str(mainprocessed_data)+'//'+subfolderscandata+'//'+'corfiles')
    corfilefolder = datfilefolder
    
    # Make sure directory exists
    os.makedirs(datfilefolder, exist_ok=True)
    
    #outputfolder = '/data/visitor/blc14894/bm32/20231007/NOBACKUP/'
    
    prefixfilename= 'ech28_'
    suffix='.tif'
    CCDLabel = 'sCMOS'
    sizeofzeropadding = 4
    print('images folder:  \n    =====> %s'%imagefolder)
    print('peaks list .dat files will be written in this folder: \n    =====> %s'%datfilefolder)
    print('peaks list .cor files will be written in this folder: \n    =====> %s'%corfilefolder)

    try:
        lltif = sorted(imagefolder.glob(f'{prefixfilename}*.tif'))
        imageindexmax = GT.getfileindex(lltif[-1])
        GT.printgreen(f'largest image index : {imageindexmax}. Set  `imageindexmax` to this value!')
    except:
        GT.printyellow(f'largest index of images file is not determined automatically... \n `imageindexmax` set to None')
    
    if Path(detfile).exists():
        GT.printgreen('calibration file exists!')
    else:
        GT.printyellow('calibration file or parameters are unknown yet! Cor files will not be created')
        
    rocessed_data.mkdir(parents=True, exist_ok=True)
    
    datfilefolder = Path(str(mainprocessed_data)+'//'+subfolderscandata+'//'+'corfiles')
    corfilefolder = datfilefolder
    
    # Make sure directory exists
    os.makedirs(datfilefolder, exist_ok=True)
    
    #outputfolder = '/data/visitor/blc14894/bm32/20231007/NOBACKUP/'
    
    prefixfilename= 'ech28_'
    suffix='.tif'
    CCDLabel = 'sCMOS'
    sizeofzeropadding = 4

#--------------------------------------------------------------

    
print('images folder:  \n    =====> %s'%imagefolder)
print('peaks list .dat files will be written in this folder: \n    =====> %s'%datfilefolder)
print('peaks list .cor files will be written in this folder: \n    =====> %s'%corfilefolder)

try:
    #lltif = sorted(imagefolder.glob(f'{prefixfilename}*.tif'))
    #imageindexmax = GT.getfileindex(lltif[-1], )
    imageindexmax = GT.get_largest_index_in_folder(imagefolder, filename_prefix=prefixfilename,
                                                         filename_suffix=suffix)
    print('largest image index in this folder:',imageindexmax)
    GT.printgreen(f'largest image index : {imageindexmax}. Set  `imageindexmax` to this value!')
except:
    GT.printyellow(f'largest index of images file is not determined automatically... \n `imageindexmax` set to None')

if Path(detfile).exists():
    GT.printgreen('calibration file exists!')
else:
    GT.printyellow('calibration file or parameters are unknown yet! Cor files will not be created')

In [ ]:
#!ls {imagefolder}

In [ ]:
detfile

In [ ]:
#32-02-812  Akamatsu et al  june 2018  Al/Al2Cu
if expId == '322812':
    imageindex = 0 # Al
    imageindex = 6 # Al2Cu
    
    maindatafolder = '/data/bm32/inhouse/data/laue/32-02-812'
    subfolderscandata = ''
    #subfolderscandata = 'ZrO2/ZrO2_ZrO2_1330C/scan0002'
    imagefolder = Path(maindatafolder+'/'+subfolderscandata)
    
    # calibration parameters file
    #'/data/bm32/inhouse/data/laue/32-02-812/calibGe111_sCMOS_3202812_vendredi_ech44.det'
    detfile = Path(maindatafolder+'/'+'calibGe111_sCMOS_3202812_vendredi_ech44.det')
    
    #mainprocessed_data = Path(maindatafolder+'//'+'PROCESSED_DATA')
    # Define the folder path starting from the home directory
    mainprocessed_data = Path("~/LaueTutorialsResults").expanduser()
    
    # Create the folder if it doesn't exist
    mainprocessed_data.mkdir(parents=True, exist_ok=True)
    
    datfilefolder = Path(str(mainprocessed_data)+'//'+subfolderscandata+'//'+'corfiles')
    corfilefolder = datfilefolder
    
    # Make sure directory exists
    os.makedirs(datfilefolder, exist_ok=True)
    
    #outputfolder = '/data/visitor/blc14894/bm32/20231007/NOBACKUP/'
    
    prefixfilename= 'ech28_'
    suffix='.tif'
    CCDLabel = 'sCMOS'
    sizeofzeropadding = 4


    
    print('images folder:  \n    =====> %s'%imagefolder)
    print('peaks list .dat files will be written in this folder: \n    =====> %s'%datfilefolder)
    print('peaks list .cor files will be written in this folder: \n    =====> %s'%corfilefolder)
    
    try:
        lltif = sorted(imagefolder.glob(f'{prefixfilename}*.tif'))
        imageindexmax = GT.getfileindex(lltif[-1])
        GT.printgreen(f'largest image index : {imageindexmax}. Set  `imageindexmax` to this value!')
    except:
        GT.printyellow(f'largest index of images file is not determined automatically... \n `imageindexmax` set to None')
    
    if Path(detfile).exists():
        GT.printgreen('calibration file exists!')
    else:
        GT.printyellow('calibration file or parameters are unknown yet! Cor files will not be created')

In [ ]:
# Cr/Zr Ribart et al 
if expId == 'blc15488':

    maindatafolder = '/data/projects/mapgrainxl/blc15488/bm32/20240601/RAW_DATA'
    subfolderscandata = '/ech15/ech15_map2Dexpo0p1sec/scan0001'
    imagefolder = Path(maindatafolder+'/'+subfolderscandata)
    
    # calibration parameters file
    detfile = Path(maindatafolder+'//'+'ech15'+'//'+'ech15_Ge'+'//'+'scan0001/calibGe.det')

    
    #mainprocessed_data = Path(maindatafolder+'//'+'PROCESSED_DATA')
    # Define the folder path starting from the home directory
    mainprocessed_data = Path("~/LaueTutorialsResults").expanduser()
    mainprocessed_data = Path(mainprocessed_data / "peaksearch")
    
    # Create the folder if it doesn't exist
    mainprocessed_data.mkdir(parents=True, exist_ok=True)
    
    datfilefolder = Path(str(mainprocessed_data)+'//'+subfolderscandata+'//'+'corfiles')
    corfilefolder = datfilefolder
    
    # Make sure directory exists
    os.makedirs(datfilefolder, exist_ok=True)
    
    #outputfolder = '/data/visitor/blc14894/bm32/20231007/NOBACKUP/'
    
    prefixfilename= 'img_'
    suffix='.tif'
    CCDLabel = 'sCMOS'
    sizeofzeropadding = 4
    print('images folder:  \n    =====> %s'%imagefolder)
    print('peaks list .dat files will be written in this folder: \n    =====> %s'%datfilefolder)
    print('peaks list .cor files will be written in this folder: \n    =====> %s'%corfilefolder)

    try:
        lltif = sorted(imagefolder.glob(f'{prefixfilename}*.tif'))
        imageindexmax = GT.getfileindex(lltif[-1])
        GT.printgreen(f'largest image index : {imageindexmax}. Set  `imageindexmax` to this value!')
    except:
        GT.printyellow(f'largest index of images file is not determined automatically... \n `imageindexmax` set to None')
    
    if detfile is not None and Path(detfile).exists():
        GT.printgreen('calibration file exists!')
    else:
        GT.printyellow('calibration file or parameters are unknown yet! Cor files will not be created')

In [ ]:
# crude & manuel way
if 0:
    imagefolder = '/home/micha/LaueProjects/Guinebretiere_Feb21/Diamond/'
    datfilefolder = '/home/micha/LaueProjects/Guinebretiere_Feb21/Diamond/datfiles'
    print('images folder:',imagefolder)
    prefixfilename= 'Gediam_'
    suffix='.tif'
    #sizeofzeropadding = 5    # image_00123.tif
    #sizeofzeropadding = 0    # image_123.tif
    sizeofzeropadding = 4
    
    print('images folder:',imagefolder)
    print('.dat files will be written in this folder: ', datfilefolder)


# [TEST] peaksearch workflow on 1 image:

- read image
- performe peaksearch
- plot image(s) with peak found
- write .dat file
- write .cor file (given geometry calibration file .det)

**select 1 image file by `imageindex`**
(splitting imagefilename allows loop over images:  prefix+index.extension)

In [ ]:
imageindex=1300

#----------------end of user input--------------------------------
paddedindex = '%s'%str(imageindex).zfill(sizeofzeropadding)
imagefilename = f"{prefixfilename}{paddedindex}{suffix}"

if not Path(imagefolder).exists():
    GT.printred('"imagefolder" does not exists!')
    
elif not (Path(imagefolder)/imagefilename).exists():
    GT.printyellow(f'this image does not exist. Check imageindex and/or imagefolder: {imagefilename} , {imagefolder}')
else:
    GT.printgreen(f"imagefilename : {imagefilename}")

**read image file and get data**



In [ ]:
print("imagefilename :",imagefilename)
im = fabio.open(Path(imagefolder)/imagefilename)
fig, ax = plt.subplots(ncols=2, figsize=(10,4))
ax[0].clear() # for lbm32gpu1
ax[1].clear()  # for lbm32gpu1
ax[0].imshow(np.log10(np.clip(im.data,a_min=0.000001, a_max=999999999999999)),vmin=0.1,vmax=2,cmap=plt.cm.OrRd)  # cmap=plt.cm.inferno)
ax[0].set_title('log scale')
ax[1].imshow(im.data,vmin=0,vmax=1000,cmap=plt.cm.OrRd)
ax[1].set_title('linear scale')
fig.suptitle(f'{imagefolder} \n {imagefilename}')
plt.show()

## PEAKSEARCH

***peaksearch*** performed on modified image 'newdataimage' (by default this is done on image given as first argument). Peaksearch results can purged from peaks already present in file as an optional argument Remove_BlackListedPeaks_fromfile.

In [ ]:
def maskpixel_in_bands(XYcam, verbose=False):
    """shift if needed pts if falling in band gaps
    XYcam is a list of 2D pts (shape = n1,2)
    return an array of 2D pts of the same shape
    """
    if verbose: print("Initial points:", XYcam)

    XYcam = np.array(XYcam)

    # Forbidden y bands
    y_bands = [(512, 549), (1062, 1099), (1612, 1649)]
    # Forbidden x bands
    x_bands = [(513, 514), (1028, 1039), (1553, 1554)]

    # Create masks for y and x bands
    y_mask = np.zeros(len(XYcam), dtype=bool)
    for band in y_bands:
        y_mask |= (XYcam[:, 1] >= band[0]) & (XYcam[:, 1] <= band[1])

    x_mask = np.zeros(len(XYcam), dtype=bool)
    for band in x_bands:
        x_mask |= (XYcam[:, 0] >= band[0]) & (XYcam[:, 0] <= band[1])

    # Count modifications
    y_modified = np.sum(y_mask)
    x_modified = np.sum(x_mask)

    # Modify y for points in forbidden y bands
    XYcam[y_mask, 1] -= 40
    # Modify x for points in forbidden x bands
    XYcam[x_mask, 0] -= 20
    
    if verbose:
        print('(y_modified, x_modified)')
        print((y_modified, x_modified))

        print("Modified points:", XYcam)

    # Return modified list
    return XYcam

In [ ]:
def removepixel_in_bands(XYcam, verbose=False):
    """
    """
    if verbose: print("Initial points:", XYcam)

    XYcam = np.array(XYcam)

    n = 11
    # Forbidden y bands
    y_bands = [[512, 549], [1062, 1099], [1612, 1649]] + np.array([[-n,+n],[-n,+n],[-n,+n]])
    # Forbidden x bands
    x_bands = [[513, 514], [1028, 1039], [1553, 1554]] + np.array([[-n,+n],[-n,+n],[-n,+n]])

    # Create masks for y and x bands
    y_mask = np.zeros(len(XYcam), dtype=bool)
    for band in y_bands:
        y_mask |= (XYcam[:, 1] >= band[0]) & (XYcam[:, 1] <= band[1])

    x_mask = np.zeros(len(XYcam), dtype=bool)
    for band in x_bands:
        x_mask |= (XYcam[:, 0] >= band[0]) & (XYcam[:, 0] <= band[1])

    inbands = np.logical_or(y_mask, x_mask)
    #print('inbands',inbands.shape)

    # Count modifications
    y_modified = np.sum(y_mask)
    x_modified = np.sum(x_mask)

    inbandsindex = np.where(inbands)
        # Return modified list
    return inbandsindex

In [ ]:
imageindex = 1300

Data_for_localMaxima = 'auto_background'
IntensityThreshold = 200
boxsize = 8
maxPixelDistanceRejection = 5
PeakSizeRange=(0.3, 10)

# --------- end of user input  ------------

paddedindex = '%s'%str(imageindex).zfill(sizeofzeropadding)
imagefilename = f"{prefixfilename}{paddedindex}{suffix}"
fullpathimagefile = Path(imagefolder)/imagefilename
print(fullpathimagefile)
#imagefilename = composite_image_filename
#imagefilename =filtered_image_filename

ti1= time.time()

#blacklistedpeaksfile=os.path.join(folder,'Blacklist.dat')
# TODO to implement in input argument Remove_BlackListedPeaks_fromfile
#blacklistedpeaksfile = None

res=RMCCD.PeakSearch(filename=os.path.join(imagefolder,imagefilename),
                     IntensityThreshold=IntensityThreshold,
                     boxsize=boxsize,
                     CCDLabel=CCDLabel,
                     fit_peaks_gaussian=1,
                    return_histo=0,
                    local_maxima_search_method=0,
                     Data_for_localMaxima=Data_for_localMaxima,
                     Saturation_value=8000000,
                     maxPixelDistanceRejection=maxPixelDistanceRejection,
                    NumberMaxofFits=10000,
                     PeakSizeRange=PeakSizeRange,
                     formulaexpression='A-1.1*B',
                     Remove_BlackListedPeaks_fromfile=None,
                     # end of most important parameters
                    xtol=0.001,
                    FitPixelDev=10,
                    center=None,
                    boxsizeROI=(200, 200),
                    PixelNearRadius=3,
                    removeedge=2,
                    thresholdConvolve=200,
                    paramsHat=(4, 5, 2),
                    verbose=0,
                    position_definition=1,
                    peakposition_definition='max',
                    write_execution_time=1,
                    Saturation_value_flatpeak=65535,
                    MinIntensity=0,
                    Fit_with_Data_for_localMaxima=False,
                    reject_negative_baseline=True,
                    listrois=None,
                     npixels=10
                    )
tps =time.time()
print(f"peak search time {np.round(tps-ti1,2)} seconds")
nbpeaks = len(res[0])
print(f"{nbpeaks} peaks  are found!")



## PLOT peaks positions in pixel units
unpack results as laue spots properties

In [ ]:
peaklist=res[0]
print(peaklist.shape)

# indexremove = removepixel_in_bands(peaklist[:,:2])
# print(len(indexremove[0]))

# peaklist = np.delete(peaklist,indexremove[0], axis=0)

print("nb of peaks in image",len(peaklist))
#peaklist[:,:2]
peak_X = peaklist[:,0]
peak_Y = peaklist[:,1]
peak_Isub = peaklist[:,2]
peak_fwaxmaj = peaklist[:,3]
peak_fwaxmin = peaklist[:,4]
peak_inclination = peaklist[:,5]
Xdev = peaklist[:,6]
Ydev = peaklist[:,7]
peak_bkg = peaklist[:,8]
Ipixmax = peaklist[:,9]
peak_Itot = peak_Isub + peak_bkg


print(peaklist.shape)

***plot the results   with zoom on the 100 first most intense peaks***

In [ ]:
fig0, ax0 = plt.subplots()

ax0.imshow(im.data,interpolation='nearest',vmin=1000,vmax=1200,cmap=plt.cm.inferno)
ax0.scatter(peak_X-1,peak_Y-1,marker='+',color='lightgreen',s=30)
ax0.set_xlim((0,2000))
ax0.set_ylim((2000,0))

fig0.suptitle('Laue pattern %s\n%d peaks'%(fullpathimagefile,len(peaklist)))

npeaks = len(peak_X)
sqrt_nvignette = min(10,int(np.ceil(np.sqrt(npeaks))))
fig = plt.figure(figsize=(sqrt_nvignette,sqrt_nvignette), )
fig.suptitle('First most intense Laue spots')
for iii in np.arange(min(npeaks,100)):
    try:
        ax = fig.add_subplot(sqrt_nvignette,sqrt_nvignette,iii+1)
        x,y = peak_X[iii]-1,peak_Y[iii]-1
        size = max(np.abs(peak_fwaxmaj[iii]),np.abs(peak_fwaxmin[iii]),3)
        intensity = peak_Itot[iii]
        xmin,xmax = int(np.round(x-1.5*size)),int(np.round(x+1.5*size))+1
        ymin,ymax = int(np.round(y-1.5*size)),int(np.round(y+1.5*size))+1
        ax.pcolormesh(np.arange(xmin-.5,xmax+.5), np.arange(ymin-.5, ymax+.5), 
                      im.data[ymin:ymax,xmin:xmax], 
                      vmin=0,vmax=intensity,cmap=plt.cm.jet)#plt.cm.inferno)
        ax.scatter(x,y,marker='+',color='lightgreen',s=30,label='fitted') # fitted
        e = Ellipse(xy=(x,y),
            width=peak_fwaxmaj[iii],
            height=peak_fwaxmin[iii],
            angle=peak_inclination[iii]+90)
        ax.add_artist(e)
        e.set_edgecolor('green')
        e.set_facecolor('none')
        ax.set_xlim(xmin,xmax-1)
        ax.set_ylim(ymax-1,ymin)
        ax.set_xticks([])
        ax.set_yticks([]) 
        #if iii == min(npeaks,100)-1:
    except Exception as e:
        print(e)
        pass
    #    plt.legend()

fig.tight_layout()
plt.show()


In [ ]:
imageindex = 1300


#imageindexmax = 650 #51*51-1

fullpath = os.path.join(imagefolder,f'{prefixfilename}%04d{suffix}'%imageindex)
#  --------- end of user input  ------------

with fabio.open(fullpath) as img:
    imgdata = img.data
#rint('imgdata.shape', imgdata.shape, '\n\n')
    
fig1,ax1 = plt.subplots()
#fig.suptitle('%s\n%s'%(ExperimentFolder,fullpath.rsplit('/RAW_DATA/')[1]))
# #ax.imshow(np.log10(imgdata), vmin = 3, vmax = 3.5, cmap=plt.cm.inferno)
ax1.imshow(imgdata, vmin = 1000, vmax =6000, cmap=plt.cm.OrRd)
# for pt in roiposition:
#     ax.scatter(pt[0],pt[1],marker='+',color='r')
    
def plotimage(imageindex=imageindex,vmin=0, vmax=1000, show_rois=False):
    # TO IMPROVE  see wrokflow JSM   laueimproc
    #print('showroi', show_rois)
    ymin, ymax = ax1.get_ylim()
    xmin, xmax = ax1.get_xlim()
    ax1.clear()
    ax1.set_xlim(xmin, xmax)
    ax1.set_ylim(ymin, ymax)
    ymin, ymax, xmin, xmax = int(ymin),int(ymax),int(xmin),int(xmax)
    if ymin>ymax:
        yminc = ymin
        ymin = ymax
        ymax = yminc
    fullpath = os.path.join(imagefolder,f'{prefixfilename}%04d{suffix}'%imageindex)
    with fabio.open(fullpath) as img:
        imgdata = img.data
        #print('imgdata.shape', imgdata.shape, '\n\n')
    if 'RAW_DATA' in fullpath:
        figtitle = '%s\n%s'%(maindatafolder,fullpath.rsplit('/RAW_DATA/')[1])
    else:
        figtitle = fullpath
    ax1.imshow(imgdata, vmin = vmin, vmax =vmax, cmap=plt.cm.jet, interpolation='nearest')
    
    if show_rois:
        #print('show rois!')
        for ptx,pty in zip(peak_X,peak_Y):
            ax1.scatter(ptx,pty,marker='+',color='r')
    ax1.set_title(figtitle)  
    plt.show()

#imageindexmax = 30000 # len(d['listindices'])-1
# imageindexmax = 4000
interactive(plotimage, imageindex=(0,imageindexmax),
            vmin=(0,1010),vmax=(1015,20000), show_rois=[False, True])

if you feel you can get more peaks from the image, you may decide to reduce `IntensityThreshold` in the `PeakSearch` function above

In [ ]:
MASK_REGION = False # True for SCMOS SCMOS_9M with some regions with weird pixels, False for sCMOS, sCMOS_4M = IMSTAR_bin#

def deletepeaks(peaklist, boxparams):
    """delete rows of peaklist array where first 2 components satisfy the box condition
    |X-Xcenter|<halfwidth & |Y-Ycenter|<halfheight 
    
    where Xcenter, Ycenter, halfwidth, halfheight = boxparams
    
    return a modified copy of peaklist"""
    
    Xcenter, Ycenter, halfwidth, halfheight = boxparams
    X,Y = peaklist[:,:2].T
    ## some peaks removal according to X, Y
    condX = np.fabs(X-Xcenter)<halfwidth
    condY = np.fabs(Y-Ycenter)<halfheight
    cond = np.logical_and(condX, condY)
    filteredpeaklist =  np.delete(peaklist, np.where(cond)[0], axis=0)
    return filteredpeaklist

#Xcenter, Ycenter, halfwidth, halfheight 
boxparams = 1009,1000,7,1100
if MASK_REGION:
    peaklist = deletepeaks(peaklist, boxparams)

## write peak list in .dat file

In [ ]:
# to add a string in the output file name
AddedString = ''

outputpeaklistprefix = prefixfilename+AddedString+str(imageindex).zfill(sizeofzeropadding)

print('peaklist.shape  (nb spots, nb properties)       :\n',peaklist.shape)
print("fullpathimagefile      :\n =====>",fullpathimagefile)
print('outputpeaklistprefix   :\n =====>', outputpeaklistprefix)

fullpath_datfile= RMCCD.writepeaklist(peaklist,outputpeaklistprefix,
                                      outputfolder=datfilefolder,
                                      initialfilename=fullpathimagefile)
datfilename = fullpath_datfile
print('Peaklist in datfilename     :\n =====>',datfilename)

## read calibration .det file and write .cor file

- set `dictcalibparams`
- .cor file (containing Laue spots porperties : X, Y, I  + 2$\theta$, $\chi$ scttering angles)

In [ ]:
dictcalibparams = None
try:
    dictcalibparams = None
    #dictcalibparams = IOLT.readCalib_det_file('/data/bm32/inhouse/LAUE/Test02Sep20_LAUE/Jeudi/calibSilukas.det')
    dictcalibparams = IOLT.readCalib_det_file(detfile)
except NameError:
    GT.printred('please provide the path to a .det file')

# write . cor if calibration parameters are known
if dictcalibparams is not None:# 
    if isinstance(dictcalibparams, dict):

        datfolder, datfile = os.path.split(fullpath_datfile)

        corfile =LaueGeo.convert2corfile(datfile,
                    [],  # should be useless and pixelsize argument is also missing...
                    dirname_in=datfolder,
                    dirname_out=datfolder,
                    CCDCalibdict=dictcalibparams,
                    add_props=True)
        print('corfile', corfile)

In [ ]:
dictcalibparams

#  PEAK SEARCH MULTIPROCESSING

PEAK SEARCH MULTIPROCESSING

- all peaks properties may also be stored in `allresults` object
- .dat and .cor files results may be written in several folders
- masks may be applied

## SET elementary peaksearch function with user defined parameters

In [ ]:
#-------------------USER INPUT------------------------------------------
# global variables
addedstring = '' # 'nb_'  # string added to all .dat filename
#CCDLabel = 'sCMOS'
#suffix = '.tif'

verboselevel = 0

if expId == 'a321220':
    formulaexpression = 'A-1.1*B'
    IntensityThreshold = 100
    boxsize = 10
    maxPixelDistanceRejection = 8
    PeakSizeRange=(0.3, 10)
if expId == '322812':
    formulaexpression = 'A-1.1*B'
    IntensityThreshold = 250
    boxsize = 10
    maxPixelDistanceRejection = 15
    PeakSizeRange=(0.1, 10)
if expId == 'blc15488':
    formulaexpression = 'A-1.1*B'
    IntensityThreshold = 250
    boxsize = 10
    maxPixelDistanceRejection = 15
    PeakSizeRange=(0.1, 10)

if expId == 'utr20':
    formulaexpression = 'A-1.1*B'
    IntensityThreshold = 100
    boxsize = 8
    maxPixelDistanceRejection = 10
    PeakSizeRange=(0.1, 10)

if expId == 'ma6758':
    formulaexpression = 'A-1.1*B'
    IntensityThreshold = 100
    boxsize = 8
    maxPixelDistanceRejection = 10
    PeakSizeRange=(0.15, 5)

#----  get rid of painful hot or bad pixels
MASK_REGION = True
#Xcenter, Ycenter, halfwidth, halfheight 
boxparams = 1009,1000,7,1100

#----------------------------------------------------------------------------
# atomic function : peaksearch on a single image
# fine parameters of RMCCD.PeakSearch can be modified here
def peaksearch_oneimage(imageindex, prefix=prefixfilename,
                               imagefolder=imagefolder,
                                outputfolder=datfilefolder,
                                sizeofzeropadding=sizeofzeropadding,
                                IntensityThreshold=IntensityThreshold,boxsize=boxsize,
                               dictcalibparams =None,
                               verbose=verboselevel):
    """
    peaksearch on single image including auto background removal, fit local intensity array without background
    
    imageindex:   int, index of image in image filename (last digits before .extension)
    prefix: str, prefix of image filename. Should end with '_'
    imagefolder: str, absolute path to image files
    outputfolder: str, absolute path to peaks list files (.dat and .cor)
    sizeofzeropadding: maximal number of digits for zero padding (usually 4)
    IntensityThreshold: int, minimum local peak amplitude"""
    
    
    
    paddedindex = '%s'%str(imageindex).zfill(sizeofzeropadding)
    imagefilename = f"{prefixfilename}{paddedindex}{suffix}"
    
    pathfile=os.path.join(imagefolder,imagefilename)
    
    try:
        # Your logic here
        
        res=RMCCD.PeakSearch(filename=pathfile,
                         IntensityThreshold=IntensityThreshold,
                         boxsize=boxsize,
                         CCDLabel=CCDLabel,
                         fit_peaks_gaussian=1,
                        return_histo=0,
                        local_maxima_search_method=0,
                         Data_for_localMaxima=Data_for_localMaxima,
                         Saturation_value=800000,
                         maxPixelDistanceRejection=maxPixelDistanceRejection,
                        NumberMaxofFits=20000,
                         PeakSizeRange=PeakSizeRange,
                         formulaexpression='A-1.1*B',
                         Remove_BlackListedPeaks_fromfile=None,
                         # end of most important parameters
                        xtol=0.001,
                        FitPixelDev=10,
                        center=None,
                        boxsizeROI=(200, 200),
                        PixelNearRadius=3,
                        removeedge=2,
                        thresholdConvolve=200,
                        paramsHat=(4, 5, 2),
                        verbose=verboselevel-1,
                        position_definition=1,
                        peakposition_definition='max',
                        write_execution_time=1,
                        Saturation_value_flatpeak=65535,
                        MinIntensity=0,
                        Fit_with_Data_for_localMaxima=False,
                        reject_negative_baseline=True,
                        listrois=None
                        )
        
        if res is None:
            return [imageindex,0]+[]
        peaklist=res[0]
        if peaklist is not None:
            nbpeaks = len(peaklist)
            
        if peaklist is not None and MASK_REGION:
            #peaklist = deletepeaks(peaklist, boxparams)
            indexremove = removepixel_in_bands(peaklist[:,:2])
            #print('MASK_REGION: nb peaks removed',len(indexremove[0]))
            
            peaklist = np.delete(peaklist,indexremove[0], axis=0)
    
            nbpeaks = len(peaklist)
    
        outputpeaklistprefix = f"{prefixfilename}{addedstring}{paddedindex}"
        fullpathimagefile = os.path.join(imagefolder,imagefilename)
    
        fullpath_datfile= RMCCD.writepeaklist(peaklist,outputpeaklistprefix,outputfolder=outputfolder,
                                        initialfilename=fullpathimagefile, verbose=verboselevel-1)
        if fullpath_datfile is None:
            return [imageindex,0]+[]
        
        if verboselevel>0:  print('fullpath_datfile',fullpath_datfile)
    #     print('len(peaklist)',len(peaklist))
        if dictcalibparams is not None:# write also .cor file
            if isinstance(dictcalibparams, dict):
    
                datfolder, datfile = os.path.split(fullpath_datfile)
    
                finalcorfilename = LaueGeo.convert2corfile(datfile,
                            [],  # should be useless and pixelsize argument is also missing...
                            dirname_in=datfolder,
                            dirname_out=datfolder,
                            CCDCalibdict=dictcalibparams,
                            add_props=True,
                            verbose=verboselevel-1)
                
                if verboselevel>0:  print('finalcorfilename',finalcorfilename)
                
        return [imageindex, nbpeaks]+peaklist.tolist()
    except Exception as e:
        print(f"Failed for {args[0]}: {str(e)}")
        return [imageindex, 0]  # 

In [ ]:
# [TEST] just to check some peak search
#dictcalibparams = None
if 1:
    ix0 = 225  #0
    for kk in range(ix0,ix0+3):
        res = peaksearch_oneimage(kk, prefix=prefixfilename, imagefolder=imagefolder, outputfolder=datfilefolder,
                                      sizeofzeropadding=sizeofzeropadding,
                                       IntensityThreshold=IntensityThreshold,
                                      boxsize=boxsize,
                                     dictcalibparams=dictcalibparams, verbose=0)
        print(f'kk = {kk}, found %d peaks'%res[1])
    GT.printgreen('Test completed')

In [ ]:
datfilefolder, dictcalibparams

In [ ]:
# uncomment to ignore warnings in the following
#import warnings
#warnings.filterwarnings('ignore')

## PEAKSEARCH on all images

In [ ]:
#imageindexmax=10000


In [ ]:
#-------------------USER INPUT----------------------------------------------
# image index coverage
firstimageindex = 0
lastimageindex = imageindexmax #46*601-1

IntensityThreshold = 100 #1000 #200
outputfolder = datfilefolder
BUILD_ALLRESULTS_LIST = True

# it is advisable to have folder with a limited nb of files (chunk size = 5000)
SPLITINTOSUBFOLDER = False
nbfiles_per_folder = 25000
subfolderprefix = 'datfiles_'

nbcpus= 64 # > 1 !!

verboselevel = 0

#--------------end of USER INPUT-----------------------------------------------
if lastimageindex is None:
    GT.printred('lastimageindex is None ie undefined. Please set to an indexinteger ')
listindices = np.arange(firstimageindex,lastimageindex+1,1).tolist()

if __name__=='__main__':
    
    if SPLITINTOSUBFOLDER:
        ll = np.array(listindices)
        # create subfolders
        nbfolders = np.amax(ll//nbfiles_per_folder)+1
        for k in range(nbfolders):
            try:
                os.mkdir(os.path.join(outputfolder,subfolderprefix+'%d_%d'%(k*nbfiles_per_folder,
                                                                 (k+1)*nbfiles_per_folder-1)))
            except FileExistsError:
                continue
    
        # ugly way to prepare the list of outputfolder
        outputfolders = []
        for _idx in ll:
            subfolder_idx = _idx // nbfiles_per_folder 
            path_to_subfolder = os.path.join(outputfolder,
                                             subfolderprefix+'%d_%d'%(subfolder_idx*nbfiles_per_folder,
                                                                 (subfolder_idx+1)*nbfiles_per_folder-1))
            outputfolders.append(path_to_subfolder)
    else:
        outputfolders = itertools.repeat(outputfolder)
    
    #---------------------Multiprocessing-----------------------------------------------
    maxnbcpus = cpu_count()
    if nbcpus is None:
        nbcpus = maxnbcpus
    else:
        nbcpus = max(2,min(nbcpus,maxnbcpus))
    #print('nbcpus', nbcpus)
    
    nbimages = len(listindices)
    #----------------------------------
    
    t00 = time.time()
    print(f'using peaksearch  on {nbimages} images, with {nbcpus} cpu(s)')
    
    args_peaksearch = zip(listindices,
                   itertools.repeat(prefixfilename),
                   itertools.repeat(imagefolder),
                   outputfolders,
                  itertools.repeat(sizeofzeropadding),
                    itertools.repeat(IntensityThreshold),
                          itertools.repeat(boxsize),
                          itertools.repeat(dictcalibparams),
                          itertools.repeat(verboselevel)
                   )
    
    with multiprocessing.Pool(nbcpus) as pool:
        if BUILD_ALLRESULTS_LIST:
            allresults = pool.starmap(peaksearch_oneimage, tqdm(args_peaksearch, total=len(listindices),
                                  desc='peak search progress bar:'), chunksize=1)
        else:
            pool.starmap(peaksearch_oneimage, tqdm(args_peaksearch, total=len(listindices),
                                  desc='peak search progress bar:'), chunksize=1)
    
    elapsedtime=time.time()-t00
    children = active_children()
    print(f'total time is {elapsedtime:.3f} sec for {nbimages} images and {nbcpus} cpu(s)')
    print(f'Active children: {len(children)}')
    
    if len(children)==0:
        GT.printgreen('\n*****************\nPeak Search is completed !!\n*****************')
    
    if BUILD_ALLRESULTS_LIST:
        GT.printgreen('"allresults" is built and contained all peaks properties...')
    
    print('Build files (.dat and .cor) are in\n =======>%s'%outputfolders)

In [ ]:
import sys
print('size of object "allresults" =====> %.2f kb '%(sys.getsizeof(allresults)/1024.))

In [ ]:
# allresults is a list. Each elem is a list: [0] image or file index, [1] nb of peaks [2:] spots properties
# allresults[15]

## pickling results

In [ ]:
# use genfolder to write output  results
import pickle

if 1: # SAVE
    dictresults={'listindices':listindices, 'datfilefolder':str(datfilefolder)}
    dictresults['allresults']=allresults
    with open(datfilefolder/'peaksprops.pickle', 'wb') as f:
        pickle.dump(dictresults, f)
        
if 0: #LOAD
    if input('Are you sure to load a previous file and overwrite current dictresults?') in ('y','yes','Y','YES','o','O'):
        with open(datfilefolder/'peaksprops.pickle', 'rb') as f:
            dictresults=pickle.load(f)
            allresults = dictresults['allresults']
            listindices = dictresults.get('listindices',None)
            datfilefolder = dictresults.get('datfilefolder', datfilefolder)
            
            if listindices is None:
                listindices=np.arange(len(allresults))
            
#print(datfilefolder, listindices, len(allresults))

# Visualisation

## PLOT nb peaks

In [ ]:
# for 2D map 

# BLISS motors command
# amesh fastaxis ... ... nbstepfast slowaxis ... ...  nbstepslow
# mapdims = (nbstepslow+1,nbstepfast+1)

if expId == 'a321220':
    mapdims = (51,51)  # slow axis,   fast axis

if expId == '322812':
    mapdims = (17,17)  # slow axis,   fast axis

if expId == 'blc15488':
    mapdims = (51,61)  # slow axis,   fast axis

# -------  data building ---------------
nbpeaks_l = np.zeros(len(listindices))

for _k in np.arange(len(listindices)):
    nbpeaks_l[_k]=allresults[_k][1]
nbpeaks_array= np.array(nbpeaks_l) 

nbpeaks2D = nbpeaks_array.reshape(mapdims)

# -------------visualisation ------------------
fig, ax = plt.subplots()
ax.imshow(nbpeaks2D, origin='lower')
ax.set_xlabel('fastaxis index')
ax.set_ylabel('slowaxis index')
if 'RAW_DATA' in str(datfilefolder):
    s1, s2 = str.split(str(datfilefolder), 'RAW_DATA')
    titleax = '%s/RAW_DATA/\n%s'%(s1,s2)
else:
    titleax = datfilefolder
    
def format_getimageindex(x, y):
    col = int(x + 0.5)
    row = int(y + 0.5)
    if col >= 0 and col < mapdims[1] and row >= 0 and row < mapdims[0]:
        img_idx = 0+ mapdims[1]*row+col
        return "x=%1.4f, y=%1.4f, imageid=%d" % (x, y,img_idx)
    else:
        return "x=%1.4f, y=%1.4f" % (x, y)
ax.format_coord = format_getimageindex

ax.set_title('%s'%titleax)

In [ ]:
# get sample map location of highest nb of peaks
fast_ix, slow_ix = np.argmax(nbpeaks2D)%mapdims[1], np.argmax(nbpeaks2D)//mapdims[1]
max_imageindex = slow_ix*mapdims[1]+fast_ix
max_nbpeaks = np.amax(nbpeaks2D)
print('max number of peaks is : %d'%max_nbpeaks)
print('at image : ', max_imageindex)
print('at scan position: fast_index, slow_index  :', fast_ix, slow_ix)


## PLOT peaks per image

Build `allpeaksXY`  big array containing all peaks coordinates of the map

In [ ]:
# build big array containing all peaks coordinates of the map
allpeakspos_l = -100*np.zeros((len(listindices),int(max_nbpeaks),2))

for _k in np.arange(len(listindices)):
    nbp = allresults[_k][1]
    if nbp > 1:
        pklist = np.array(allresults[_k][2:])
        _n = len(pklist)
        allpeakspos_l[_k][:_n]=pklist[:,:2]
allpeaksXY= np.array(allpeakspos_l) 

In [ ]:
# to see one element of `allpeaksXY`
if 0:
    local_index = 6
    fig, ax = plt.subplots()
    spots= allpeaksXY[local_index]
    ax.scatter(spots[:,0], spots[:,1])
    ax.set_xlim(0,2050)
    ax.set_ylim(2050,0)
    ax.set_title('Peaks: imageindex=%d'%listindices[local_index])

### GUI PLOT peak list Browser

In [ ]:
imageindex0 = 0  # initial value for the first plot

from ipywidgets import interact, interactive, fixed, interact_manual
fig, ax = plt.subplots()
spots= allpeaksXY[imageindex0]
ax.scatter(spots[:,0], spots[:,1])
ax.set_xlim(0,2000)
ax.set_ylim(2000,0)

def plotpeaks(imageindex=imageindex0, shownbpeaks=True, grid=False):
    ymin, ymax = ax.get_ylim()
    xmin, xmax = ax.get_xlim()
    ax.clear()
    ax.set_xlim(xmin, xmax)
    ax.set_ylim(ymin, ymax)
    
    spots= allpeaksXY[imageindex]
    print(len(spots))
    ax.scatter(spots[:,0], spots[:,1])
    if 'RAW_DATA' in str(datfilefolder):
        s1, s2 = str.split(str(datfilefolder), 'RAW_DATA')
        titleax = '%s/RAW_DATA/\n%s'%(s1,s2)
    else:
        titleax = str(datfilefolder)
    titleax +='\nimageindex=%d'%imageindex
    if shownbpeaks:
        titleax+='\nnbpeaks=%d'%len(spots[:,0])
    if grid:
        ax.grid()
    ax.set_title(titleax)

    
interactive(plotpeaks, imageindex=(0,max(listindices)-1), shownbpeaks=[True, False], grid=[False, True])

In [ ]:
# as others do, for small set of data: we could do pca or nmf to find main grains (see sergio)
# good for small number of grains